In [1]:
# Run this in a JupyterLab notebook inside the Docker environment
import sys
print(f"Python: {sys.version}")
print(f"Running inside: {sys.prefix}")
print("Course environment is ready!")

Python: 3.11.10 | packaged by conda-forge | (main, Oct 16 2024, 01:19:04) [GCC 13.3.0]
Running inside: /opt/conda
Course environment is ready!


# Task 2.1: Transaction producer

In [7]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime
 
producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
 
shops = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
cathegories = ['elektronika', 'odzież', 'żywność', 'książki']
 
def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(shops),
        'category': random.choice(cathegories),
        'timestamp': datetime.now().isoformat(),
    }
 
 
for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(1)
 
producer.flush()
producer.close()

Overwriting producer.py


# Task 3.1: Filter large transactions

In [8]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Listening for large transactions (amount > 1000)...")

# YOUR CODE
# For each message: check if amount > 1000, if so print an alert
# Format: ALERT: TX0042 | 2345.67 PLN | Warsaw | electronics

for message in consumer:
    tx = message.value
    if tx.get('amount', 0) > 1000:
        print(f"ALERT: {tx.get('tx_id')} | {tx.get('amount')}| {tx.get('store')} | {tx.get('category')}")

Overwriting consumer_filter.py


# Task 3.2 Transform and enrich

In [9]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

# YOUR CODE
# Read from 'transactions' (use a DIFFERENT group_id!)
# Add risk_level field based on amount
# Print enriched transaction
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='enriched-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Risk level:")

for message in consumer:
    tx = message.value
    if tx.get('amount', 0) > 1000:
        print(f"MEDIUM | {tx.get('tx_id')} | {tx.get('amount')}| {tx.get('store')} | {tx.get('category')}")
    elif tx.get('amount', 0) > 3000:
        print(f"HIGH | {tx.get('tx_id')} | {tx.get('amount')}| {tx.get('store')} | {tx.get('category')}")
    else:
        print(f"LOW | {tx.get('tx_id')} | {tx.get('amount')}| {tx.get('store')} | {tx.get('category')}")

Writing consumer_enrich.py


# Task 4.1 Count transactions per store

In [11]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

# YOUR CODE
# For each message:
#   1. Increment store_counts[store]
#   2. Add amount to total_amount[store]
#   3. Every 10 messages, print a summary table:
#      Store | Count | Total Amount | Avg Amount

for message in consumer:
    tx = message.value
    store = tx.get('store')
    amount = tx.get('amount', 0)

    store_counts[store] += 1

    total_amount[store] = total_amount.get(store, 0) + amount

    msg_count += 1

    if msg_count % 10 == 0:
        print(f"\n--- SUMMARY REPORT (Total messages: {msg_count}) ---")
        print(f"{'Store':<15} | {'Count':<8} | {'Total Amount':<15} | {'Avg Amount':<10}")
        print("-" * 60)
        
        for s in store_counts:
            count = store_counts[s]
            total = total_amount[s]
            avg = total / count
            print(f"{s:<15} | {count:<8} | {total:<12.2f} PLN | {avg:<10.2f} PLN")
        print("-" * 60)

Overwriting consumer_count.py


# Task 4.2: Stats per category

In [12]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

# YOUR CODE
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = {}
msg_count = 0

print("Stats:")

for message in consumer:
    tx = message.value
    cat = tx.get('category')
    amount = tx.get('amount', 0)

    if cat not in stats:
        stats[cat] = {
            'count': 0, 
            'total': 0, 
            'min': float('inf'), 
            'max': float('-inf')
        }

    stats[cat]['count'] += 1
    stats[cat]['total'] += amount
    if amount < stats[cat]['min']: stats[cat]['min'] = amount
    if amount > stats[cat]['max']: stats[cat]['max'] = amount
    
    msg_count += 1

    if msg_count % 10 == 0:
        print(f"\n{'='*85}")
        print(f"{'Category':<15} | {'Count':<6} | {'Min':<10} | {'Max':<10} | {'Avg':<10} | {'Total':<12}")
        print(f"{'-'*85}")
        
        for cat_name, data in stats.items():
            avg = data['total'] / data['count']
            print(f"{cat_name:<15} | {data['count']:<6} | {data['min']:<10.2f} | {data['max']:<10.2f} | {avg:<10.2f} | {data['total']:<12.2f}")
        print(f"{'='*85}")

Writing consumer_stats.py


# Homework: Consumer velocity

In [13]:
%%file consumer_velocity.py
from kafka import KafkaConsumer
import json
from datetime import datetime, timedelta

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='velocity-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_history = {}

print("Velocity Detection...")

for message in consumer:
    tx = message.value
    user_id = tx.get('user_id')
    
    current_time = datetime.fromisoformat(tx.get('timestamp'))
    
    if user_id not in user_history:
        user_history[user_id] = []
    
    user_history[user_id].append(current_time)
    
    sixty_seconds_ago = current_time - timedelta(seconds=60)
    user_history[user_id] = [t for t in user_history[user_id] if t > sixty_seconds_ago]
    
    if len(user_history[user_id]) > 3:
        print(f"⚠️  VELOCITY ALERT: User {user_id} made {len(user_history[user_id])} transactions in 60 seconds!")
        print(f"   Detailed: {tx['tx_id']} | {tx['amount']} PLN | {tx['store']}")
        print("-" * 50)


Writing consumer_velocity.py
